<a href="https://www.kaggle.com/code/iamadityak/ohid-1-unet?scriptVersionId=313949243" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# U-Net v3 — OHID-1 

## Cell 1 — Install & Imports

In [ ]:
!pip install -q rasterio

import os, json, time, warnings, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import rasterio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    confusion_matrix, classification_report
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {total_mem:.1f} GB')
print('Imports done ✓')

## Cell 2 — Configuration

In [ ]:
IMAGE_DIR  = '/kaggle/input/datasets/iamadityak/ohid-1/hrnavy-OHID-1-220e32e/images'
LABEL_DIR  = '/kaggle/input/datasets/iamadityak/ohid-1/hrnavy-OHID-1-220e32e/labels'
RESULT_DIR = '/kaggle/working/results/'
CKPT_DIR   = '/kaggle/working/ckpts/'
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

N_SCENES   = 10
N_BANDS    = 32
N_CLASSES  = 7
IMG_SIZE   = 512

CLASS_NAMES  = ['Building','Farmland','Forest',
                'Road','Water','Bareland','FishShed']
CLASS_COLORS = ['#000000','#e6194b','#3cb44b','#228b22',
                '#808080','#4169e1','#f58231','#911eb4']

# ── KEY MEMORY SETTINGS ───────────────────────────────────
PATCH_SIZE        = 11      # smaller patch = less memory per sample
                            # 11×11 still gives good spatial context
N_TRAIN_PER_CLASS = 500     # same as all paper baselines
BATCH_SIZE        = 128     # safe for GPU with 11×11×32 patches

# ── Training ──────────────────────────────────────────────
EPOCHS       = 150
LR           = 3e-4
LR_MIN       = 1e-6
WEIGHT_DECAY = 1e-4
PATIENCE     = 25
N_TRIALS     = 3            # 3 trials per scene (faster than 10)

print('Config ✓')
print(f'Patch size  : {PATCH_SIZE}×{PATCH_SIZE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Train/class : {N_TRAIN_PER_CLASS}')

# Memory estimate per scene
# Max labeled pixels ≈ 200k, store only coordinates (2 ints)
# = 200k × 2 × 4 bytes = 1.6 MB per scene → FINE
print(f'\nRAM per scene (coords only): ~{200000*2*4/1e6:.1f} MB ✓')

## Cell 3 — Load Scenes (Images + Labels Only)
### ⚠️ No patch extraction here — only store images & coordinates

In [ ]:
def load_scene(scene_id):
    """Load & normalize one scene. Returns float32 image, int64 label."""
    img_path = os.path.join(IMAGE_DIR, f'201912_{scene_id}.tif')
    lbl_path = os.path.join(LABEL_DIR, f'201912_{scene_id}.tif')

    with rasterio.open(img_path) as f:
        image = f.read().astype(np.float32)   # (32, 512, 512)
    with rasterio.open(lbl_path) as f:
        label = f.read(1).astype(np.int64)    # (512, 512)

    # Per-band z-score normalization
    for b in range(image.shape[0]):
        mu, sigma  = image[b].mean(), image[b].std() + 1e-8
        image[b]   = (image[b] - mu) / sigma

    return image, label


def get_pixel_coords(label):
    """
    Return row, col coordinates of ALL labeled pixels.
    Stores only indices — NOT patches.
    Memory: N_pixels × 2 ints ≈ tiny.
    """
    rows, cols = np.where(label > 0)
    targets    = label[rows, cols] - 1   # 0-indexed
    return rows.astype(np.int16), cols.astype(np.int16), targets.astype(np.int8)


def stratified_split_coords(rows, cols, targets,
                            n_train=N_TRAIN_PER_CLASS, seed=42):
    """Split pixel coordinates 80/20 per class."""
    np.random.seed(seed)
    tr_idx, te_idx = [], []

    for cls in np.unique(targets):
        idx    = np.where(targets == cls)[0]
        n      = min(n_train, int(0.2 * len(idx)))
        chosen = np.random.choice(idx, n, replace=False)
        rest   = np.setdiff1d(idx, chosen)
        tr_idx.extend(chosen.tolist())
        te_idx.extend(rest.tolist())

    tr_idx = np.array(tr_idx)
    te_idx = np.array(te_idx)

    return (rows[tr_idx], cols[tr_idx], targets[tr_idx],
            rows[te_idx], cols[te_idx], targets[te_idx])


# ── Load all scenes ────────────────────────────────────────
# Only images + coordinate arrays stored
# Total RAM: 10 × (32×512×512×4 bytes) + coords
#           = 10 × 33.6 MB + ~20 MB = ~356 MB → SAFE
print('Loading all 10 scenes (images + coordinates only)...')
ALL_DATA = []

for sid in range(1, N_SCENES + 1):
    t0             = time.time()
    image, label   = load_scene(sid)
    rows, cols, tgts = get_pixel_coords(label)

    (tr_r, tr_c, tr_t,
     te_r, te_c, te_t) = stratified_split_coords(
                             rows, cols, tgts,
                             n_train=N_TRAIN_PER_CLASS,
                             seed=SEED)

    ALL_DATA.append({
        'scene_id' : sid,
        'image'    : image,     # (32,512,512) float32 ~33MB
        'label'    : label,     # (512,512) int64
        # Train coords
        'tr_r': tr_r, 'tr_c': tr_c, 'tr_t': tr_t,
        # Test coords
        'te_r': te_r, 'te_c': te_c, 'te_t': te_t,
    })

    cls_dist = {CLASS_NAMES[c]: int((tgts==c).sum())
                for c in np.unique(tgts)}
    print(f'  Scene {sid:2d} | '
          f'labeled={len(rows):6,} | '
          f'train={len(tr_r):5,} | '
          f'test={len(te_r):6,} | '
          f'{time.time()-t0:.1f}s')
    print(f'          dist={cls_dist}')

# Memory report
import sys
total_mb = sum(
    sys.getsizeof(d['image']) + sys.getsizeof(d['label'])
    for d in ALL_DATA
) / 1e6
print(f'\nTotal RAM used (approx): {total_mb:.0f} MB ✓')
print('All scenes loaded ✓')

## Cell 4 — Class Weights

In [ ]:
def compute_class_weights(all_data, n_classes=N_CLASSES):
    counts = np.zeros(n_classes, dtype=np.float64)
    for d in all_data:
        for c in range(n_classes):
            counts[c] += (d['tr_t'] == c).sum()
    weights = 1.0 / (counts + 1e-8)
    weights = weights / weights.mean()
    return torch.FloatTensor(weights)


CLASS_WEIGHTS = compute_class_weights(ALL_DATA)

# Plot
all_counts = np.zeros(N_CLASSES)
for d in ALL_DATA:
    for c in range(N_CLASSES):
        all_counts[c] += (d['label'] == c+1).sum()
total = all_counts.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = [CLASS_COLORS[i+1] for i in range(N_CLASSES)]

bars = axes[0].bar(CLASS_NAMES, all_counts/1e3,
                   color=colors, edgecolor='black')
axes[0].set_title('Pixel Count per Class (all 10 scenes)')
axes[0].set_ylabel('Pixels (thousands)')
axes[0].tick_params(axis='x', rotation=30)
for bar, cnt in zip(bars, all_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+1,
                 f'{100*cnt/total:.1f}%', ha='center', fontsize=8)

axes[1].bar(CLASS_NAMES, CLASS_WEIGHTS.numpy(),
            color=colors, edgecolor='black')
axes[1].axhline(1.0, color='red', linestyle='--', label='weight=1')
axes[1].set_title('Class Weights (Inverse Frequency)')
axes[1].set_ylabel('Weight')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/class_weights.png', dpi=150)
plt.show()

print('Class weights:')
for n, w in zip(CLASS_NAMES, CLASS_WEIGHTS.numpy()):
    print(f'  {n:12s}: {w:.4f}')

## Cell 5 — On-the-Fly Patch Dataset


In [ ]:
class CoordPatchDataset(Dataset):
    """
    Memory-efficient dataset.

    Stores:
      - image_padded : (32, 512+pad, 512+pad) — one scene
      - rows, cols   : pixel coordinates
      - targets      : class labels

    On each __getitem__ call:
      1. Look up (row, col)
      2. Slice patch from padded image
      3. Apply augmentation
      4. Return (patch_tensor, label)

    RAM: only ONE padded image held at a time
    (per DataLoader worker → managed by PyTorch)
    """
    def __init__(self, image, rows, cols, targets,
                 patch_size=PATCH_SIZE, augment=False,
                 oversample_minority=True):

        self.patch_size = patch_size
        self.augment    = augment
        pad             = patch_size // 2

        # Pad once — stored as single array
        self.image_pad  = np.pad(
            image,
            ((0,0),(pad,pad),(pad,pad)),
            mode='reflect'
        ).astype(np.float32)   # (32, 512+2*pad, 512+2*pad)

        self.rows    = rows.astype(np.int16)
        self.cols    = cols.astype(np.int16)
        self.targets = targets.astype(np.int64)

        # Oversample minority classes in training
        if augment and oversample_minority:
            MINORITY = [5, 6]   # Bareland, FishShed (0-indexed)
            FACTOR   = 8        # replicate 8× to balance
            ex_r, ex_c, ex_t = [], [], []
            for cls in MINORITY:
                idx = np.where(self.targets == cls)[0]
                if len(idx) > 0:
                    rep = np.tile(idx, FACTOR)
                    ex_r.append(self.rows[rep])
                    ex_c.append(self.cols[rep])
                    ex_t.append(self.targets[rep])
            if ex_r:
                self.rows    = np.concatenate(
                    [self.rows]    + ex_r)
                self.cols    = np.concatenate(
                    [self.cols]    + ex_c)
                self.targets = np.concatenate(
                    [self.targets] + ex_t)

            # Shuffle
            perm         = np.random.permutation(len(self.rows))
            self.rows    = self.rows[perm]
            self.cols    = self.cols[perm]
            self.targets = self.targets[perm]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r, c   = int(self.rows[idx]), int(self.cols[idx])
        P      = self.patch_size

        # Extract patch (FAST — numpy slice)
        patch  = self.image_pad[:, r:r+P, c:c+P].copy()  # (32,P,P)
        target = int(self.targets[idx])

        if self.augment:
            # Random flips
            if np.random.rand() > 0.5:
                patch = patch[:, :, ::-1].copy()
            if np.random.rand() > 0.5:
                patch = patch[:, ::-1, :].copy()
            # Random 90° rotation
            k = np.random.randint(0, 4)
            if k > 0:
                patch = np.rot90(patch, k, axes=(1,2)).copy()
            # Spectral noise (random bands)
            if np.random.rand() > 0.6:
                nb    = np.random.choice(N_BANDS, 4, replace=False)
                patch[nb] += np.random.normal(
                    0, 0.05, patch[nb].shape).astype(np.float32)

        return (torch.from_numpy(patch),
                torch.tensor(target, dtype=torch.long))


# Sanity check — ONLY scene 0 in memory
d0  = ALL_DATA[0]
ds_ = CoordPatchDataset(
    d0['image'], d0['tr_r'], d0['tr_c'], d0['tr_t'],
    augment=True)
p_, t_ = ds_[0]
print(f'Patch shape : {p_.shape}  (32, {PATCH_SIZE}, {PATCH_SIZE})')
print(f'Target      : {t_} → {CLASS_NAMES[int(t_)]}')
print(f'Train set size (with oversampling): {len(ds_):,}')

# Memory of padded image
pad_mb = ds_.image_pad.nbytes / 1e6
print(f'Padded image RAM : {pad_mb:.1f} MB per scene ✓')
del ds_
print('CoordPatchDataset ✓')

## Cell 6 — U-Net Architecture

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, max(ch//r, 8)),
            nn.ReLU(inplace=True),
            nn.Linear(max(ch//r, 8), ch),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
        self.skip = (nn.Conv2d(in_ch, out_ch, 1, bias=False)
                     if in_ch != out_ch else nn.Identity())
        self.se   = SEBlock(out_ch)

    def forward(self, x):
        return self.se(self.net(x)) + self.skip(x)


class PatchUNet(nn.Module):
    """
    Input : (B, 32, P, P)  where P = PATCH_SIZE = 11
    Output: (B, 7)         class logits for center pixel

    Encoder: 11→5→2 (with maxpool)
    Decoder: 2→5→11 (with upsample + skip)
    Head   : global avg pool → linear
    """
    def __init__(self, in_ch=N_BANDS, n_cls=N_CLASSES, base=64):
        super().__init__()

        # Spectral embedding: 32 → base
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.GELU()
        )

        # Encoder
        self.enc1  = ResBlock(base,   base*2)   # (B,128,11,11)
        self.pool1 = nn.MaxPool2d(2)             # (B,128,5,5)
        self.enc2  = ResBlock(base*2, base*4)   # (B,256,5,5)
        self.pool2 = nn.MaxPool2d(2)             # (B,256,2,2)

        # Bottleneck
        self.bot   = ResBlock(base*4, base*8)   # (B,512,2,2)

        # Decoder
        self.up2   = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec2  = ResBlock(base*8, base*4)   # after cat
        self.up1   = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec1  = ResBlock(base*4, base*2)

        # Head
        self.head  = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base*2, base),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(base, n_cls)
        )

        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,
                    mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):              # (B,32,11,11)
        x  = self.stem(x)              # (B,64,11,11)
        s1 = self.enc1(x)              # (B,128,11,11)
        s2 = self.enc2(self.pool1(s1)) # (B,256,5,5)
        b  = self.bot(self.pool2(s2))  # (B,512,2,2)

        d2 = self.up2(b)               # (B,256,4,4)
        d2 = F.interpolate(d2, size=s2.shape[2:],
                           mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, s2], 1))  # (B,256,5,5)

        d1 = self.up1(d2)              # (B,128,10,10)
        d1 = F.interpolate(d1, size=s1.shape[2:],
                           mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, s1], 1))  # (B,128,11,11)

        return self.head(d1)           # (B, 7)


# Verify
m_  = PatchUNet().to(DEVICE)
n_p = sum(p.numel() for p in m_.parameters())
d_  = torch.zeros(4, N_BANDS, PATCH_SIZE, PATCH_SIZE).to(DEVICE)
with torch.no_grad(): o_ = m_(d_)
print(f'Parameters  : {n_p:,} ({n_p/1e6:.2f}M)')
print(f'Output shape: {o_.shape}')    # (4, 7)
del m_, d_; torch.cuda.empty_cache()

## Cell 7 — Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss with class weights + label smoothing.
    Aggressively handles class imbalance.
    """
    def __init__(self, weights, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing
        self.register_buffer('weights', weights)

    def forward(self, logits, targets):
        C   = logits.size(1)
        # Label smoothing
        with torch.no_grad():
            smooth = torch.full_like(logits,
                                     self.smoothing / (C - 1))
            smooth.scatter_(1, targets.unsqueeze(1),
                            1.0 - self.smoothing)

        log_p   = F.log_softmax(logits, dim=1)
        p       = torch.exp(log_p)
        p_t     = (p * F.one_hot(targets, C).float()).sum(1)
        focal_w = (1 - p_t).pow(self.gamma)
        alpha   = self.weights[targets]
        ce      = -(smooth * log_p).sum(1)
        return (alpha * focal_w * ce).mean()


CRITERION = FocalLoss(
    weights   = CLASS_WEIGHTS.to(DEVICE),
    gamma     = 2.0,
    smoothing = 0.1
).to(DEVICE)

print('Focal Loss ✓  (gamma=2, label_smoothing=0.1)')
print(f'Class weights: {CLASS_WEIGHTS.numpy().round(3)}')

## Cell 8 — Metrics & Confusion Matrix

In [ ]:
def compute_metrics(y_true, y_pred, n_classes=N_CLASSES):
    OA    = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    cm    = confusion_matrix(y_true, y_pred,
                             labels=list(range(n_classes)))
    # Per-class accuracy (diagonal / row sum)
    row_sum = cm.sum(axis=1)
    per_cls = np.where(row_sum > 0,
                       cm.diagonal() / (row_sum + 1e-8),
                       0.0)
    AA = per_cls.mean()
    return {'OA':    round(float(OA),    4),
            'AA':    round(float(AA),    4),
            'Kappa': round(float(kappa), 4)}


def plot_confusion_matrix(y_true, y_pred, scene_id,
                          class_names, save_dir=RESULT_DIR):
    present = sorted(np.unique(
        np.concatenate([y_true, y_pred])).tolist())
    names   = [class_names[i] for i in present]

    cm      = confusion_matrix(y_true, y_pred, labels=present)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True)+1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Confusion Matrix — Scene {scene_id}', fontsize=13)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=axes[0])
    axes[0].set_title('Counts')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
    axes[0].tick_params(axis='x', rotation=35)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=names, yticklabels=names,
                vmin=0, vmax=1, ax=axes[1])
    axes[1].set_title('Normalised (row = true class)')
    axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
    axes[1].tick_params(axis='x', rotation=35)

    plt.tight_layout()
    path = os.path.join(save_dir, f'cm_scene{scene_id}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    return names, cm, cm_norm


@torch.no_grad()
def predict_batched(model, dataset, batch_size=2048, device=DEVICE):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=2,
                        pin_memory=True)
    preds = []
    for xb, _ in loader:
        preds.extend(model(xb.to(device)).argmax(1).cpu().tolist())
    return np.array(preds)


print('Metrics & confusion matrix utilities ✓')

## Cell 9 — Training Function

In [ ]:
def train_scene(data, scene_id, trial=0, device=DEVICE):
    """
    Train one scene for one trial.
    Memory efficient: only one padded scene in RAM per DataLoader.
    """
    # ── Datasets built from coordinate arrays ─────────────
    train_ds = CoordPatchDataset(
        data['image'], data['tr_r'], data['tr_c'], data['tr_t'],
        augment=True,  oversample_minority=True)
    test_ds  = CoordPatchDataset(
        data['image'], data['te_r'], data['te_c'], data['te_t'],
        augment=False, oversample_minority=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True,
        drop_last=True)

    # ── Model ─────────────────────────────────────────────
    torch.manual_seed(SEED + trial * 7)
    model     = PatchUNet().to(device)
    optimizer = optim.AdamW(model.parameters(),
                            lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(
                    optimizer, T_0=50, T_mult=1, eta_min=LR_MIN)

    ckpt      = os.path.join(CKPT_DIR,
                             f'unet_s{scene_id}_t{trial}.pt')
    best_oa   = 0.0
    best_pred = None
    no_imp    = 0
    history   = {'loss':[], 'OA':[], 'AA':[], 'Kappa':[]}

    for epoch in range(1, EPOCHS + 1):
        # Train
        model.train()
        ep_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = CRITERION(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
        scheduler.step()

        # Evaluate every 5 epochs
        if epoch % 5 == 0 or epoch == 1:
            y_pred = predict_batched(model, test_ds)
            m      = compute_metrics(data['te_t'], y_pred)

            history['loss'].append(ep_loss/len(train_loader))
            history['OA'].append(m['OA'])
            history['AA'].append(m['AA'])
            history['Kappa'].append(m['Kappa'])

            print(f'  Ep {epoch:3d} | '
                  f'loss={ep_loss/len(train_loader):.4f} | '
                  f'OA={m["OA"]:.4f} | '
                  f'AA={m["AA"]:.4f} | '
                  f'K={m["Kappa"]:.4f}',
                  end='\n' if epoch % 25 == 0 else '\r')

            if m['OA'] > best_oa:
                best_oa   = m['OA']
                best_pred = y_pred.copy()
                no_imp    = 0
                torch.save(model.state_dict(), ckpt)
            else:
                no_imp += 1

            if no_imp >= PATIENCE // 5:
                print(f'\n  Early stop ep {epoch}')
                break

    # Load best weights
    model.load_state_dict(torch.load(ckpt))
    best_m = compute_metrics(data['te_t'], best_pred)
    print(f'\n  ✓ Best OA={best_m["OA"]:.4f} '
          f'AA={best_m["AA"]:.4f} '
          f'K={best_m["Kappa"]:.4f}')

    return model, best_m, best_pred, test_ds, history


print('Training function ✓')

## Cell 10 — Main Experiment: All 10 Scenes

In [ ]:
ALL_RESULTS   = []
ALL_PREDS     = []
ALL_HISTORIES = []

for scene_idx, data in enumerate(ALL_DATA):
    sid = data['scene_id']
    print(f'\n{"="*60}')
    print(f'  SCENE {sid:2d} / {N_SCENES}  |  '
          f'train={len(data["tr_r"]):,}  '
          f'test={len(data["te_r"]):,}')
    print(f'{"="*60}')

    trial_metrics = []
    trial_preds   = []
    best_model    = None
    best_oa_ever  = 0.0
    best_pred_ever = None

    for trial in range(N_TRIALS):
        print(f'\n  --- Trial {trial+1}/{N_TRIALS} ---')
        model, metrics, y_pred, test_ds, hist = train_scene(
            data, sid, trial=trial)

        trial_metrics.append(metrics)
        trial_preds.append(y_pred)
        ALL_HISTORIES.append(
            {'scene': sid, 'trial': trial, **hist})

        if metrics['OA'] > best_oa_ever:
            best_oa_ever   = metrics['OA']
            best_pred_ever = y_pred.copy()
            best_model     = model

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # Mean over trials
    mean_OA    = np.mean([m['OA']    for m in trial_metrics])
    mean_AA    = np.mean([m['AA']    for m in trial_metrics])
    mean_Kappa = np.mean([m['Kappa'] for m in trial_metrics])
    std_OA     = np.std( [m['OA']    for m in trial_metrics])

    result = {
        'model'  : 'UNet_v3',
        'scene'  : sid,
        'OA'     : round(mean_OA,    4),
        'AA'     : round(mean_AA,    4),
        'Kappa'  : round(mean_Kappa, 4),
        'OA_std' : round(std_OA,     4),
        'all_OA' : [m['OA'] for m in trial_metrics]
    }
    ALL_RESULTS.append(result)
    ALL_PREDS.append(best_pred_ever)

    # Save per-scene result
    with open(os.path.join(RESULT_DIR,
              f'UNet_v3_scene{sid}.json'),'w') as f:
        json.dump(result, f, indent=2)

    print(f'\n  Scene {sid} → '
          f'OA={mean_OA:.4f}±{std_OA:.4f}  '
          f'AA={mean_AA:.4f}  '
          f'Kappa={mean_Kappa:.4f}')

print(f'\n{"="*60}')
print('ALL SCENES DONE ✓')
print(f'{"="*60}')

## Cell 11 — Confusion Matrices for All 10 Scenes

In [ ]:
print('Generating confusion matrices...\n')

for data, y_pred, result in zip(ALL_DATA, ALL_PREDS, ALL_RESULTS):
    sid = data['scene_id']
    print(f'Scene {sid} — OA={result["OA"]:.4f}')

    names, cm, cm_norm = plot_confusion_matrix(
        data['te_t'], y_pred,
        scene_id=sid,
        class_names=CLASS_NAMES
    )

    # Print per-class accuracy
    per_cls = cm.diagonal() / (cm.sum(axis=1) + 1e-8)
    print('  Per-class accuracy:')
    for name, acc in zip(names, per_cls):
        bar = '█' * int(acc * 20)
        print(f'    {name:12s}: {acc:.4f}  {bar}')
    print()

print('Confusion matrices saved ✓')

## Cell 12 — Classification Maps

In [ ]:
from matplotlib.colors import ListedColormap
CMAP = ListedColormap(CLASS_COLORS)

fig, axes = plt.subplots(3, N_SCENES, figsize=(40, 12))
fig.suptitle('U-Net v3 — All 10 OHID-1 Scenes',
             fontsize=18, y=1.01)

for idx, (data, y_pred, result) in enumerate(
        zip(ALL_DATA, ALL_PREDS, ALL_RESULTS)):
    sid   = data['scene_id']
    image = data['image']
    label = data['label']
    H, W  = label.shape

    # Reconstruct prediction map from test coords
    pred_map          = np.zeros(H*W, dtype=np.int32)
    flat_mask         = label.reshape(-1) > 0
    
    
    all_coords_flat   = data['te_r'].astype(np.int32) * W + data['te_c'].astype(np.int32)
    pred_map[all_coords_flat] = y_pred + 1   # 1-indexed
    
   
    tr_coords_flat    = data['tr_r'].astype(np.int32) * W + data['tr_c'].astype(np.int32)
    pred_map[tr_coords_flat] = data['tr_t'].astype(np.int32) + 1
    
    pred_map = pred_map.reshape(H, W)

    # True color
    rgb = np.stack([image[13], image[7], image[2]], axis=-1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    rgb = np.clip(rgb, 0, 1)

    axes[0,idx].imshow(rgb)
    axes[0,idx].set_title(f'Scene {sid}', fontsize=9)
    axes[0,idx].axis('off')

    axes[1,idx].imshow(label, cmap=CMAP, vmin=0, vmax=7)
    axes[1,idx].axis('off')

    axes[2,idx].imshow(pred_map, cmap=CMAP, vmin=0, vmax=7)
    axes[2,idx].set_title(f'OA={result["OA"]:.3f}', fontsize=8)
    axes[2,idx].axis('off')

for row, txt in enumerate(['True Color','Ground Truth','U-Net v3']):
    axes[row,0].set_ylabel(txt, fontsize=11)

legend_p = [mpatches.Patch(color=CLASS_COLORS[i+1],
                            label=CLASS_NAMES[i])
            for i in range(N_CLASSES)]
fig.legend(handles=legend_p, loc='lower center', ncol=7,
           fontsize=11, bbox_to_anchor=(0.5,-0.04))
plt.tight_layout()
plt.savefig('/kaggle/working/maps_all_scenes.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('Maps saved ✓')


## Cell 13 — Summary Table & Comparison Plot

In [ ]:
rows = [{'Scene' : f'Scene {r["scene"]}',
         'OA'    : r['OA'],
         'OA_std': r['OA_std'],
         'AA'    : r['AA'],
         'Kappa' : r['Kappa']}
        for r in ALL_RESULTS]
df = pd.DataFrame(rows)
mean_row = pd.DataFrame([{
    'Scene':'MEAN',
    'OA'   : round(df['OA'].mean(),    4),
    'OA_std': round(df['OA_std'].mean(),4),
    'AA'   : round(df['AA'].mean(),    4),
    'Kappa': round(df['Kappa'].mean(), 4)
}])
df_full = pd.concat([df, mean_row], ignore_index=True)

print('='*55)
print('  U-NET v3 — OHID-1 RESULTS')
print('='*55)
print(df_full.to_string(index=False))
print('='*55)

paper = {
    '1DCNN':{'OA':0.592,'AA':0.616,'Kappa':0.501},
    '2DCNN':{'OA':0.663,'AA':0.687,'Kappa':0.583},
    '3DCNN':{'OA':0.615,'AA':0.648,'Kappa':0.529},
    'CDCNN':{'OA':0.724,'AA':0.562,'Kappa':0.653},
    'SVM'  :{'OA':0.691,'AA':0.543,'Kappa':0.614},
    'DBMA' :{'OA':0.736,'AA':0.601,'Kappa':0.670},
    'DBDA' :{'OA':0.740,'AA':0.610,'Kappa':0.675},
    'HyLITE':{'OA':0.869,'AA':0.698,'Kappa':0.798},
    'SSSAN':{'OA':0.895,'AA':0.836,'Kappa':0.855},
    'CVSSN':{'OA':0.903,'AA':0.886,'Kappa':0.864},
}
our = df_full[df_full['Scene']=='MEAN'].iloc[0]
paper['U-Net v3 (Ours)'] = {
    'OA':our['OA'], 'AA':our['AA'], 'Kappa':our['Kappa']}

comp = pd.DataFrame(paper).T.sort_values('OA')
print('\nVs paper baselines:')
print(comp.to_string())

df_full.to_csv('/kaggle/working/UNet_v3_results.csv', index=False)
comp.to_csv('/kaggle/working/comparison.csv')

# Final JSON
with open('/kaggle/working/UNet_v3_final.json','w') as f:
    json.dump({'model':'UNet_v3',
               'mean_OA':float(our['OA']),
               'mean_AA':float(our['AA']),
               'mean_Kappa':float(our['Kappa']),
               'per_scene':[{'scene':r['scene'],
                              'OA':r['OA'],
                              'AA':r['AA'],
                              'Kappa':r['Kappa']}
                             for r in ALL_RESULTS]},
              f, indent=2)

# ── Plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('OHID-1 — All Models (Mean over 10 scenes)',
             fontsize=14)

for ax, metric in zip(axes, ['OA','AA','Kappa']):
    vals    = comp[metric].values
    models  = comp.index.tolist()
    colors_ = ['#ff4444' if 'Ours' in m else
               '#ffd700' if m in ['CVSSN','SSSAN','HyLITE']
               else '#4dabf7' for m in models]
    bars = ax.barh(models, vals, color=colors_,
                   edgecolor='black', height=0.6)
    for bar, v in zip(bars, vals):
        ax.text(v+0.003, bar.get_y()+bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)
    ax.set_xlim(0, 1.08)
    ax.set_title(metric, fontsize=13)
    ax.axvline(0.9, color='green', linestyle='--', alpha=0.4)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(color='#ff4444', label='Our U-Net v3'),
    Patch(color='#ffd700', label='Top paper methods'),
    Patch(color='#4dabf7', label='Other baselines')],
    loc='lower center', ncol=3, fontsize=10,
    bbox_to_anchor=(0.5,-0.06))
plt.tight_layout()
plt.savefig('/kaggle/working/final_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\n FINAL: OA={our["OA"]:.4f}  '
      f'AA={our["AA"]:.4f}  '
      f'Kappa={our["Kappa"]:.4f}')
print('Notebook complete ✓')